# Tutorial 02 (Bonds to Devices) — The bond is a spring

A chemical bond behaves like a stiff spring: stretch it and it pulls back, let go and it vibrates at a fixed frequency. This notebook works out that frequency for hydrogen, shows how Raman scattering of laser light reveals it, and predicts how much slower the heavier isotope deuterium vibrates. All inputs are **measured** spectroscopic constants and line positions taken from the literature; the spring and Raman calculations built on them are simple models. No new measurements are introduced.

This is Episode 2 of the *Bonds to Devices* series. The short video for the same episode is made by `make_reel_ep02.py` in this folder, from the same `bond_spring.py` module, so every number in the video can be traced to a cell below.

## Learning objectives

By the end of the notebook, you should be able to:

- turn a vibrational wavenumber (cm⁻¹) into a frequency, a period and a photon energy;
- compute the stiffness of a bond from its vibration and explain why H₂ and D₂ share the same spring;
- predict the wavelength of Raman-scattered light for a given laser and bond;
- predict an isotope's vibration from the reduced mass, and explain why the ratio is close to, but not exactly, √2; and
- say why H₂'s vibration is seen by Raman scattering and not by ordinary infrared absorption.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import bond_spring as bs

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")

## 1. Data source and units

Spectroscopists describe a vibration by its wavenumber ν̃ in cm⁻¹ (the number of wavelengths of the matching light per centimetre). Two kinds of measured numbers are used. The **constants** ωₑ, ωₑxₑ and ωₑyₑ describe the shape of the energy levels; they come from Huber & Herzberg (1979) via the NIST Chemistry WebBook. The **measured fundamentals** are the actual spacing between the lowest two vibrational levels, measured by Dickenson et al. (2013) to better than 0.001 cm⁻¹. The D₂ and HD fundamentals are kept aside to test the isotope prediction.

In [ ]:
constants = pd.DataFrame({
    "quantity": ["omega_e (H2)", "omega_e x_e (H2)", "omega_e y_e (H2)",
                 "fundamental H2", "fundamental HD (test)", "fundamental D2 (test)",
                 "omega_e (D2) (check only)"],
    "value_cm1": [bs.OMEGA_E_H2_CM1, bs.OMEGA_E_XE_H2_CM1, bs.OMEGA_E_YE_H2_CM1,
                  bs.FUNDAMENTAL_H2_CM1, bs.FUNDAMENTAL_HD_CM1, bs.FUNDAMENTAL_D2_CM1,
                  bs.OMEGA_E_D2_CM1],
    "source": ["Huber & Herzberg (1979)"] * 3 + ["Dickenson et al. (2013)"] * 3
              + ["Huber & Herzberg (1979)"],
})
masses = pd.DataFrame({
    "molecule": ["H2", "HD", "D2"],
    "reduced_mass_amu": [bs.MU_H2_AMU, bs.MU_HD_AMU, bs.MU_D2_AMU],
})
display(constants)
masses

## 2. How fast does the bond swing?

The frequency is the wavenumber times the speed of light, ν = c ν̃. The period is its inverse. Converting to energy with E = hc ν̃ gives the size of one vibrational step.

In [ ]:
nu_H2_cm1 = bs.FUNDAMENTAL_H2_CM1
frequency_H2_Hz = bs.frequency_Hz(nu_H2_cm1)
period_H2_fs = bs.period_fs(nu_H2_cm1)
quantum_H2_eV = nu_H2_cm1 * bs.EV_PER_CM1

print(f"Frequency: {frequency_H2_Hz:.4e} Hz  (about {frequency_H2_Hz / 1e12:.0f} trillion per second)")
print(f"One swing: {period_H2_fs:.2f} fs")
print(f"One vibrational step: {quantum_H2_eV:.4f} eV")

## 3. The stiffness of the spring

For a mass μ on a spring of stiffness k, the angular frequency is √(k/μ). For a diatomic molecule, μ is the reduced mass, and the harmonic constant ωₑ (not the measured fundamental) describes the ideal spring at the bottom of the energy curve. So k = μ(2πcωₑ)². The same formula applied to D₂ tests whether both molecules have the same spring, as they should: they have the same electrons and the same energy curve.

In [ ]:
k_H2_N_per_m = bs.force_constant_N_per_m(bs.OMEGA_E_H2_CM1, bs.MU_H2_AMU)
k_D2_N_per_m = bs.force_constant_N_per_m(bs.OMEGA_E_D2_CM1, bs.MU_D2_AMU)
print(f"k for H2: {k_H2_N_per_m:.1f} N/m")
print(f"k for D2: {k_D2_N_per_m:.1f} N/m")
print(f"Difference: {100 * (k_D2_N_per_m / k_H2_N_per_m - 1):+.2f} %")

## 4. Why the measured note is lower than ωₑ

A real bond is not a perfect spring: it gets softer as it stretches. The level spacing therefore shrinks a little with each step, and the first spacing is G(1) − G(0) = ωₑ − 2ωₑxₑ + (13/4)ωₑyₑ, not ωₑ. The plot shows the first few levels of the ideal spring and of the real bond, both from the H₂ constants.

In [ ]:
v = np.arange(0, 6)
n = v + 0.5
levels_harmonic_cm1 = bs.OMEGA_E_H2_CM1 * n
levels_real_cm1 = (bs.OMEGA_E_H2_CM1 * n - bs.OMEGA_E_XE_H2_CM1 * n**2
                   + bs.OMEGA_E_YE_H2_CM1 * n**3)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(v, levels_harmonic_cm1, "o--", label="perfect spring (omega_e only)")
ax.plot(v, levels_real_cm1, "s-", label="real H2 (with anharmonicity)")
ax.set_xlabel("Vibrational quantum number v")
ax.set_ylabel("Energy above the curve minimum (cm$^{-1}$)")
ax.set_title("H2 vibrational levels")
ax.legend()
fig.tight_layout()
plt.show()

spacing_from_constants_cm1 = bs.fundamental_from_constants_cm1(
    bs.OMEGA_E_H2_CM1, bs.OMEGA_E_XE_H2_CM1, bs.OMEGA_E_YE_H2_CM1)
print(f"G(1) - G(0) from the constants: {spacing_from_constants_cm1:.2f} cm^-1")
print(f"Measured fundamental:           {nu_H2_cm1:.2f} cm^-1")

## 5. Seeing the vibration with light

H₂ has two identical atoms, so stretching it does not create an electric dipole. Ordinary infrared absorption needs a changing dipole and so barely sees H₂ (only very weak quadrupole lines exist). Raman scattering works differently: the light distorts the electron cloud, and a scattered photon can leave behind one vibrational step. It then comes out with less energy, so a longer wavelength: 1/λ_out = 1/λ_in − ν̃. A green 532 nm laser is used here.

In [ ]:
laser_nm = bs.LASER_NM
raman = pd.DataFrame({
    "molecule": ["H2", "HD", "D2"],
    "shift_cm1": [bs.FUNDAMENTAL_H2_CM1, bs.FUNDAMENTAL_HD_CM1, bs.FUNDAMENTAL_D2_CM1],
})
raman["scattered_nm"] = bs.stokes_wavelength_nm(laser_nm, raman["shift_cm1"]).round(1)
raman["photon_energy_eV"] = bs.photon_energy_eV(raman["scattered_nm"]).round(3)
print(f"Laser: {laser_nm:.0f} nm, photon energy {bs.photon_energy_eV(laser_nm):.3f} eV")
raman

Only a tiny fraction of the scattered light is shifted, and almost all of it is shifted to the red. The blue-shifted (anti-Stokes) line needs molecules that are already vibrating, and the Boltzmann factor says how few there are at room temperature.

In [ ]:
fraction_excited = bs.anti_stokes_to_stokes_boltzmann(nu_H2_cm1, 298.15)
print(f"Fraction of H2 already in v = 1 at 25 C: {fraction_excited:.1e}")

## 6. Heavier atoms, lower note

Deuterium has the same electron and nuclear charge as hydrogen but about twice the mass. The energy curve, and so the spring, is the same. Only the reduced mass changes, by a factor of almost exactly 2, so a perfect spring would vibrate √2 times slower. The function `bs.isotope_fundamental_cm1` goes further and scales all three H₂ constants with the mass, then predicts the fundamental of HD and D₂. Those measured values were not used to build the prediction.

In [ ]:
mass_ratio = bs.MU_D2_AMU / bs.MU_H2_AMU
ratio_measured = bs.FUNDAMENTAL_H2_CM1 / bs.FUNDAMENTAL_D2_CM1
print(f"Reduced-mass ratio D2/H2: {mass_ratio:.4f}  ->  perfect-spring ratio {np.sqrt(mass_ratio):.4f}")
print(f"Measured ratio of fundamentals H2/D2: {ratio_measured:.4f}")

prediction = pd.DataFrame({
    "molecule": ["HD", "D2"],
    "reduced_mass_amu": [bs.MU_HD_AMU, bs.MU_D2_AMU],
    "predicted_cm1": bs.isotope_fundamental_cm1(np.array([bs.MU_HD_AMU, bs.MU_D2_AMU])),
    "measured_cm1": [bs.FUNDAMENTAL_HD_CM1, bs.FUNDAMENTAL_D2_CM1],
})
prediction["difference_percent"] = 100 * (prediction["predicted_cm1"] / prediction["measured_cm1"] - 1)
prediction.round(3)

In [ ]:
mu_grid_amu = np.linspace(0.4, 1.1, 200)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(mu_grid_amu, bs.harmonic_wavenumber_cm1(k_H2_N_per_m, mu_grid_amu),
        "--", label="perfect spring, k from H2")
ax.plot(mu_grid_amu, bs.isotope_fundamental_cm1(mu_grid_amu), label="H2 constants scaled by mass")
ax.plot([bs.MU_H2_AMU, bs.MU_HD_AMU, bs.MU_D2_AMU],
        [bs.FUNDAMENTAL_H2_CM1, bs.FUNDAMENTAL_HD_CM1, bs.FUNDAMENTAL_D2_CM1],
        "ko", label="measured (Dickenson et al. 2013)")
ax.set_xlabel("Reduced mass (amu)")
ax.set_ylabel("Fundamental vibration (cm$^{-1}$)")
ax.set_title("Hydrogen isotopes: heavier means lower")
ax.legend()
fig.tight_layout()
plt.show()

## 7. Validation checks

Each check compares the model with a number it was not built from, or with a value that must hold by construction.

In [ ]:
# The H2 constants must reproduce the measured H2 fundamental to better than 0.01 %.
assert abs(spacing_from_constants_cm1 / nu_H2_cm1 - 1) < 1e-4

# H2 and D2 must have the same spring (same energy curve) to within 0.5 %.
assert abs(k_D2_N_per_m / k_H2_N_per_m - 1) < 5e-3

# Textbook stiffness of H2 is about 575 N/m.
assert 570 < k_H2_N_per_m < 580

# The isotope prediction must match the measured HD and D2 lines (not used as input) to 0.05 %.
assert (prediction["difference_percent"].abs() < 0.05).all()

# The measured ratio sits below sqrt(2) because the real bond is softer than a perfect spring.
assert ratio_measured < np.sqrt(mass_ratio)

# Energy bookkeeping: laser photon = scattered photon + one vibrational step.
out_H2_nm = bs.stokes_wavelength_nm(laser_nm, nu_H2_cm1)
assert np.isclose(bs.photon_energy_eV(laser_nm),
                  bs.photon_energy_eV(out_H2_nm) + quantum_H2_eV, rtol=1e-9)

print("All checks passed.")

## Interpretation and checks

- **Calculation.** The H₂ bond vibrates at 4161 cm⁻¹, which is 1.25 × 10¹⁴ swings per second, or one swing every 8.0 fs. Its stiffness is about 575 N/m, and D₂ gives the same value to within 0.2 %.
- **Raman.** A 532 nm photon that leaves one vibrational step (0.52 eV) behind in H₂ comes out at 683 nm, deep red. D₂ shifts it to 633 nm. This is consistent with Raman spectra of hydrogen, and is how the vibration of a molecule without a dipole is measured.
- **Isotopes.** Scaling the H₂ constants by the reduced mass predicts the HD and D₂ fundamentals to within about 0.03 %, without using those measured values. The ratio of H₂ to D₂ is 1.39, a little below √2 = 1.41, because anharmonicity lowers the H₂ note more than the D₂ note.
- **What this does not show.** The remaining 0.03 % is not noise: it comes from small effects beyond the single shared energy curve (the Born–Oppenheimer approximation), which the simple mass scaling ignores. The spring picture works well for the lowest levels and fails near the bond-breaking limit, where the levels crowd together.
- **"Fixed note".** Rotation splits the Raman line into a few components a few cm⁻¹ apart (Q(0), Q(1), …). The 4161 cm⁻¹ value is the J = 0 line.

## Exercise

1. Repeat the Raman table for a 785 nm laser, which is common in Raman microscopes. Is the H₂ line still visible to the eye?
2. Tritium (T, mass 3.01605 amu) makes T₂. Use `bs.reduced_mass_amu` and `bs.isotope_fundamental_cm1` to predict its fundamental. The measured value is about 2464.5 cm⁻¹; how close do you get?
3. Use `bs.anti_stokes_to_stokes_boltzmann` to find the temperature at which 1 % of H₂ molecules are already vibrating.
4. Open-ended: the video says the bond "rings at one fixed note". Write two sentences for a non-scientist that explain why a heavier ball on the same spring gives a lower note, using an everyday example.

## References

- Dickenson, G. D., Niu, M. L., Salumbides, E. J., Komasa, J., Eikema, K. S. E., Pachucki, K., & Ubachs, W. (2013). Fundamental vibration of molecular hydrogen. *Physical Review Letters*, 110, 193601.
- Huber, K. P., & Herzberg, G. (1979). *Molecular Spectra and Molecular Structure IV: Constants of Diatomic Molecules*. Van Nostrand Reinhold. Values for H₂ and D₂ as listed in the NIST Chemistry WebBook, SRD 69.
- Stoicheff, B. P. (1957). High resolution Raman spectroscopy of gases. IX. Spectra of H₂, HD, and D₂. *Canadian Journal of Physics*, 35, 730–741.
- Ferraro, J. R., Nakamoto, K., & Brown, C. W. (2003). *Introductory Raman Spectroscopy* (2nd ed.). Academic Press.
- Atkins, P., & de Paula, J. (2014). *Atkins' Physical Chemistry* (10th ed.). Oxford University Press. Chapter on vibrational spectroscopy of diatomic molecules.
- Tiesinga, E., Mohr, P. J., Newell, D. B., & Taylor, B. N. (2021). CODATA recommended values of the fundamental physical constants: 2018. *Reviews of Modern Physics*, 93, 025010.
- Trivikram, T. M., Schlösser, M., Ubachs, W., & Salumbides, E. J. (2018). Relativistic and QED effects in the fundamental vibration of T₂. *Physical Review Letters*, 120, 163002. (T₂ value used in the exercise.)
- Wang, M., Huang, W. J., Kondev, F. G., Audi, G., & Naimi, S. (2021). The AME 2020 atomic mass evaluation (II). *Chinese Physics C*, 45, 030003.